## Remove probable duplicated samples from the combined metadata

We take only one of each, choosen at random.

In [1]:
import os
import pandas as pd

## Settings

In [2]:
SAVE_RESULTS = True
OUTPUT_DIR = "../metadata/2026-04-05_sample-duplication/"

## Load data

In [3]:
df_dups = pd.read_csv(f"{OUTPUT_DIR}/duplicates.csv", dtype=dict(sample_id=str))
df_all = pd.read_csv("../metadata/2026-01-23_consolidated-metadata/table.zambia_all_data_combined.csv", dtype=dict(sample_id=str))

In [4]:
print(f"""
    We have {df_all.shape[0]} samples in the aggregated metadata.
    We have {df_dups.shape[0]} potentially duplicated samples.
    They are from {len(df_dups.cluster.unique())} distinct duplication clusters.
    Of the duplicated IDs, {len(set(df_dups.sample_id).intersection(df_all.sample_id))} are found in the aggregated metadata.
""")


    We have 9528 samples in the aggregated metadata.
    We have 73 potentially duplicated samples.
    They are from 17 distinct duplication clusters.
    Of the duplicated IDs, 73 are found in the aggregated metadata.



## Identify duplicates to remove

In [5]:
keep_samples = []
for cluster, dfc in df_dups.groupby("cluster"):
    k = dfc.sample(1, random_state=123)['sample_id'].iloc[0]
    print(f"Cluster {cluster} has {dfc.shape[0]} samples, keeping {k}.")
    keep_samples.append(k)

Cluster 0 has 5 samples, keeping 01039031.
Cluster 1 has 2 samples, keeping 4043511.
Cluster 8 has 2 samples, keeping 01039190.
Cluster 16 has 3 samples, keeping 1012732.
Cluster 22 has 2 samples, keeping 8021909.
Cluster 30 has 2 samples, keeping 8011425.
Cluster 31 has 2 samples, keeping 8011443.
Cluster 32 has 12 samples, keeping 8031240.
Cluster 33 has 2 samples, keeping 8031233.
Cluster 39 has 14 samples, keeping 8021776.
Cluster 40 has 3 samples, keeping 8021802.
Cluster 41 has 2 samples, keeping 8021947.
Cluster 42 has 2 samples, keeping 8021951.
Cluster 43 has 2 samples, keeping 8021952.
Cluster 46 has 9 samples, keeping 8030166.
Cluster 48 has 2 samples, keeping 8034056.
Cluster 49 has 7 samples, keeping 8035607.


In [6]:
remove_samples = list(set(df_dups.sample_id).difference(keep_samples))

In [7]:
print(f"Overall, keeping {len(keep_samples)} and removing {len(remove_samples)} samples.")

Overall, keeping 17 and removing 56 samples.


In [8]:
set(keep_samples).intersection(remove_samples)

set()

In [9]:
df_dups_removed = df_dups.query("sample_id in @remove_samples")

In [10]:
df_all_dedup = df_all.query("sample_id not in @remove_samples")

In [11]:
df_all_dedup.shape[0], df_all.shape[0]

(9472, 9528)

In [12]:
df_all_dedup.shape[0] == df_all.shape[0] - len(remove_samples)

True

- Good, we have removed each duplicate
- Now we save and run

### Write

In [13]:
if SAVE_RESULTS:
    df_all_dedup.to_csv(f"{OUTPUT_DIR}/table.zambia_all_data_combined.dedup.csv", index=False)
    df_dups_removed.to_csv(f"{OUTPUT_DIR}/table.removed_duplicates.csv", index=False)

In [14]:
df_all_dedup.columns

Index(['study', 'sample_id', 'province', 'district', 'ward', 'healthfac',
       'lat', 'long', 'collection_date', 'rdt_result', 'parasitemia',
       'extraction_id', 'take_for_sequencing', 'screening_method', 'has_pcr',
       'location'],
      dtype='object')

In [15]:
df_all_dedup

,study,sample_id,province,district,ward,healthfac,lat,long,collection_date,rdt_result,parasitemia,extraction_id,take_for_sequencing,screening_method,has_pcr,location
0,MIS2024,8010001,central,kapiri mposhi,chibwelelo,NaN,-14.102388,28.649287,2024-04-18,pf_pos,NaN,DV380,False,pet_pcr,True,chibwelelo
1,MIS2024,8010002,central,kapiri mposhi,chibwelelo,NaN,-14.102388,28.649287,2024-04-18,pf_pos,40277.0,DV381,True,pet_pcr,True,chibwelelo
2,MIS2024,8010003,central,kapiri mposhi,chibwelelo,NaN,-14.102388,28.649287,2024-04-18,pf_pos,NaN,DV382,False,pet_pcr,True,chibwelelo
3,MIS2024,8010004,central,kapiri mposhi,chibwelelo,NaN,-14.102388,28.649287,2024-04-18,pf_pos,NaN,DV383,False,pet_pcr,True,chibwelelo
4,MIS2024,8010005,central,kapiri mposhi,chibwelelo,NaN,-14.102388,28.649287,2024-04-18,pf_neg,NaN,NaN,NaN,NaN,False,chibwelelo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9522,HRP23,8021822,central,kapiri mposhi,NaN,mutaba health centre,-13.921303,28.673837,NaN,pf_neg,NaN,NaN,NaN,NaN,False,mutaba health centre
9523,HRP23,8021821,central,kapiri mposhi,NaN,mutaba health centre,-13.921303,28.673837,NaN,pf_neg,NaN,NaN,NaN,NaN,False,mutaba health centre
9524,HRP23,8021802,central,kapiri mposhi,NaN,mutaba health centre,-13.921303,28.673837,NaN,pf_pos,NaN,DR748,True,pet_pcr,True,mutaba health centre
9525,HRP23,8021804,central,kapiri mposhi,NaN,mutaba health centre,-13.921303,28.673837,NaN,pf_pos,NaN,DR750,False,pet_pcr,True,mutaba health centre


## Create a table by study

In [16]:
DIR_SEQDATA = "../seqdata/all_standard/summaries/HRP23_MIS2024"

In [17]:
df_samples = (pd.read_csv("../seqdata/all/summaries/HRP23_MIS2024/summary.samples_amplicons_qc.csv", dtype=dict(sample_id=str))
              .query("name == 'kelch13-p383-727'"))

In [18]:
(df_samples.status == 'passing').sum() # good!

np.int64(1950)

In [19]:
df_combined = pd.merge(left=df_all_dedup, right=df_samples, on="sample_id", validate="1:1")

In [20]:
from itertools import product

In [21]:
def formatted_crosstable(rows, columns) -> pd.DataFrame:
    """
    Create a formatted crosstable
    """
    _a = pd.crosstab(rows, columns, margins=True)
    _b = pd.crosstab(rows, columns, normalize='index', margins=True) * 100
    _c = _a.copy()

    _c = _a.copy()
    for c in _c.columns:
        _c[c] = _c[c].astype(str)
    for ix, c in product(_a.index, _a.columns):
        if c == 'All':
            continue
        _c.loc[ix, c] = f"{_a.loc[ix, c]} ({_b.loc[ix, c]:.0f})"

    return _c

In [22]:
def clean_tables(df: pd.DataFrame) -> pd.DataFrame:
    df = df.reset_index()
    df.columns.name = None
    df.index.name = None

    cmap = {
        "study": "Study",
        "province": "Province",
        "failing": "Failing – n (%)",
        "passing": "Passing – n (%)",
        "All": "Overall – n"
    }
    if "province" in df.columns:
        df["province"] = df["province"].str.capitalize()
    df = df.rename(cmap, axis=1)
    
    
    return df

In [23]:
cdf_study = formatted_crosstable(df_combined["study"],
                                 df_combined["status"])
cdf_province = formatted_crosstable([df_combined["study"], df_combined["province"]],
                                    df_combined["status"])

In [24]:
cdf_province = clean_tables(cdf_province)
cdf_study = clean_tables(cdf_study)

In [25]:
cdf_total = pd.concat([cdf_province.query("Study != 'All'"), cdf_study])

In [26]:
cdf_total.Province = ['All' if pd.isna(p) else p for p in cdf_total.Province]

In [27]:
cdf_total

,Study,Province,Failing – n (%),Passing – n (%),Overall – n
0,HRP23,Central,33 (23),110 (77),143
1,HRP23,Copperbelt,35 (13),239 (87),274
2,HRP23,Eastern,45 (11),347 (89),392
3,HRP23,Luapula,93 (20),377 (80),470
4,HRP23,Lusaka,5 (6),82 (94),87
5,HRP23,Muchinga,10 (7),138 (93),148
6,HRP23,Northern,2 (4),49 (96),51
7,HRP23,Northwestern,34 (19),146 (81),180
8,HRP23,Western,40 (18),179 (82),219
9,MIS2024,Central,11 (37),19 (63),30


In [28]:
cdf_total.to_excel(f"../tables/stable_study-inclusion.xlsx", index=False)